# Chapter 3. 나이브베이즈 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter03_3_naivebayes.ipynb)

책 본문: [3.3 나이브베이즈: 독립을 가정하고 차원의 저주를 피한다](https://smhanlab.com/book-ml/kor/ml1/chapter03/3.html)

이 노트북은 책 3.3절의 나이브베이즈 스팸 필터를 코드로 실행해서 **본문의 손 계산값이 실제로 맞는지** 직접 검증합니다. 다항 나이브베이즈 분류(학습에서 보지 못한 메일 + margin), 2-단어 장난감 모델의 손 계산 표, 라플라스 스무딩이 없는 경우의 "0 vs 0" 문제, 사전확률이 판단을 뒤집는 "동점 메일", `0.98^200000` 언더플로우, 그리고 나이브베이즈 vs GDA 파라미터 수 비교까지 끝까지 실행됩니다.


## 1. 다항 나이브베이즈: 스팸 8통·정상 8통, 어휘 26개

본문 "실습" 섹션의 학습 데이터입니다 — 스팸 8통과 정상 8통, 그리고
**학습에서 보지 못한** 단어 조합 4개를 테스트 메일로 씁니다.


In [1]:
import math

def train_multinomial_nb(emails, labels):
    vocab = sorted(set(w for e in emails for w in e))
    counts, nwords, ndocs = {0: {}, 1: {}}, {0: 0, 1: 0}, {0: 0, 1: 0}
    for e, l in zip(emails, labels):
        ndocs[l] += 1
        for w in e:                       # 이번엔 등장 횟수까지 센다
            counts[l][w] = counts[l].get(w, 0) + 1
            nwords[l] += 1
    return {"vocab": vocab, "counts": counts,
            "nwords": nwords, "ndocs": ndocs, "n_total": len(labels)}

spam_train = ["free money now", "win prize today", "free cash click",
              "prize free offer", "money win now", "free click here",
              "prize cash deal", "offer win cheap"]
ham_train = ["meeting project tomorrow", "project deadline review",
             "team meeting report", "meeting notes call", "project update plan",
             "report review friday", "team lunch call", "meeting agenda notes"]

emails = spam_train + ham_train
labels = [1]*8 + [0]*8
emails = [e.split() for e in emails]   # 토큰화: 단어(문자) 단위로 센다
model = train_multinomial_nb(emails, labels)
print(f"어휘 크기 V = {len(model['vocab'])}   (본문 표기: 26)")


어휘 크기 V = 26   (본문 표기: 26)


In [2]:
def multinomial_log_probs(email, model):
    out = {}
    for l in (0, 1):
        lp = math.log(model["ndocs"][l] / model["n_total"])   # log P(y)
        for w in email:
            p = (model["counts"][l].get(w, 0) + 1) / \
                (model["nwords"][l] + len(model["vocab"]))    # 라플라스
            lp += math.log(p)                                  # log 가능도 합
        out[l] = lp
    return out

test_emails = [
    ("free prize money", 1),        # 학습에서 보지 못한 스팸 조합
    ("win click offer", 1),
    ("meeting project report", 0),  # 학습에서 보지 못한 정상 조합
    ("team call notes", 0),
]
n_correct = 0
for email, truth in test_emails:
    lp = multinomial_log_probs(email.split(), model)
    pred = 1 if lp[1] > lp[0] else 0
    n_correct += pred == truth
    print(f"{'SPAM' if truth else 'HAM ':4s} pred={pred} "
          f"margin={abs(lp[1]-lp[0]):.2f} nats  {email!r}")
print(f"\n{n_correct}/4 올바르게 분류")


SPAM pred=1 margin=4.09 nats  'free prize money'
SPAM pred=1 margin=3.58 nats  'win click offer'
HAM  pred=0 margin=4.09 nats  'meeting project report'
HAM  pred=0 margin=3.30 nats  'team call notes'

4/4 올바르게 분류


## 2. 손으로 한 번: 2-단어 어휘 {공짜, 회의}

본문 "손으로 한 번"의 학습 데이터:
스팸 `{공짜}`, `{공짜, 회의}` / 정상 `{회의}`, `{회의}`.
베르누이 나이브베이즈(등장 여부만) + 라플라스 스무딩으로 네 가지
메일의 \(P(\text{spam}|x)\)가 본문의 표(0.857 / 0.667 / 0.182 / 0.400)와
맞는지 확인합니다.


In [3]:
VOCAB = ["공짜", "회의"]
spam_docs = [{"공짜"}, {"공짜", "회의"}]
ham_docs  = [{"회의"}, {"회의"}]

def bernoulli_logprob(words, docs, vocab=VOCAB):
    """sum_j log P(x_j | 클래스) — 라플라스 스무딩이 든 베르누이 버전."""
    n_docs = len(docs)
    lp = 0.0
    for w in vocab:
        n_present = sum(1 for d in docs if w in d)
        p_present = (n_present + 1) / (n_docs + 2)
        lp += math.log(p_present) if w in words else math.log(1 - p_present)
    return lp

cases = [
    ({"공짜"},          0.857),
    ({"공짜", "회의"},   0.667),
    ({"회의"},          0.182),
    (set(),             0.400),
]
for words, expected in cases:
    lp_spam = math.log(0.5) + bernoulli_logprob(words, spam_docs)
    lp_ham  = math.log(0.5) + bernoulli_logprob(words, ham_docs)
    p_spam = 1 / (1 + math.exp(lp_ham - lp_spam))
    label = " ".join(sorted(words)) if words else "(어떤 단어도 없음)"
    ok = "OK" if abs(p_spam - expected) < 0.001 else "MISMATCH"
    print(f"{label:>28s}  P(spam|x) = {p_spam:.3f}   (본문 표: {expected})  {ok}")


                          공짜  P(spam|x) = 0.857   (본문 표: 0.857)  OK
                       공짜 회의  P(spam|x) = 0.667   (본문 표: 0.667)  OK
                          회의  P(spam|x) = 0.182   (본문 표: 0.182)  OK
                 (어떤 단어도 없음)  P(spam|x) = 0.400   (본문 표: 0.4)  OK


## 3. 라플라스 스무딩: 학습에서 한 번도 본 적 없는 "당첨"

같은 학습 데이터(어휘 {공짜, 회의})로 만든 모델에, **학습에서 한 번도
보지 못한 단어 "당첨"**이 들어 있는 메일 `{공짜, 당첨}`이 오면 —
스무딩 없이 하면 두 클래스의 가능도가 모두 정확히 0이 됩니다.
스무딩을 넣으면 "당첨"은 두 클래스 모두 1/4(중립 신호)이 되어
"공짜"의 3:1 증거가 판단을 결정합니다.


In [4]:
def bernoulli_raw(words, docs):
    """스무딩 없는 가능도: 메일에 나온 단어들의 P(x_j|클래스) 곱."""
    n_docs = len(docs)
    prod = 1.0
    for w in words:
        n_present = sum(1 for d in docs if w in d)
        prod *= n_present / n_docs
    return prod

mail = {"공짜", "당첨"}
print("스무딩 없이:  P(x|spam) =", bernoulli_raw(mail, spam_docs),
      "  P(x|ham) =", bernoulli_raw(mail, ham_docs),
      "   <- 0 vs 0, 비교 자체가 불가능")

VOCAB2 = VOCAB + ["당첨"]   # 새 단어는 학습에서 0회 등장
lp_spam = bernoulli_logprob(mail, spam_docs, VOCAB2)
lp_ham  = bernoulli_logprob(mail, ham_docs, VOCAB2)
p_spam = 1 / (1 + math.exp(lp_ham - lp_spam))
print(f"스무딩 넣으면:  P(spam|x) = {p_spam:.3f}   가능도비 = "
      f"{math.exp(lp_spam - lp_ham):.0f}:1  (당첨은 1/4 vs 1/4로 중립)")


스무딩 없이:  P(x|spam) = 0.0   P(x|ham) = 0.0    <- 0 vs 0, 비교 자체가 불가능
스무딩 넣으면:  P(spam|x) = 0.857   가능도비 = 6:1  (당첨은 1/4 vs 1/4로 중립)


`{공짜, 회의}`는 가능도비가 정확히 2(스팸 쪽)입니다(\(\tfrac34 \times
\tfrac12 = \tfrac38\) vs \(\tfrac14 \times \tfrac34 = \tfrac3{16}\)).
가능도비가 2처럼 "동점 쪽"으로 쏠린 메일에서는 분류를 사전확률이 좌우합니다
— 50% → 0.667, 90% → 0.947, 10% → 0.182(스팸이 10%만 되면 9:1의
사전확률 우위가 2:1의 단어 증거를 누르고 예측이 정상으로 뒤집힘).

In [5]:
words = {"공짜", "회의"}
llr = bernoulli_logprob(words, spam_docs) - bernoulli_logprob(words, ham_docs)
print(f"가능도비 = {math.exp(llr):.4f}   <- 정확히 2:1 스팸 (3/4·1/2 vs 1/4·3/4)")
for prior in (0.5, 0.9, 0.1):
    lp_spam = math.log(prior) + bernoulli_logprob(words, spam_docs)
    lp_ham  = math.log(1 - prior) + bernoulli_logprob(words, ham_docs)
    p_spam = 1 / (1 + math.exp(lp_ham - lp_spam))
    print(f"스팸 사전확률 {prior:3.0%}  ->  P(spam|x) = {p_spam:.3f}")


가능도비 = 2.0000   <- 정확히 2:1 스팸 (3/4·1/2 vs 1/4·3/4)
스팸 사전확률 50%  ->  P(spam|x) = 0.667
스팸 사전확률 90%  ->  P(spam|x) = 0.947
스팸 사전확률 10%  ->  P(spam|x) = 0.182


## 5. 왜 로그를 더하는가: 0.98^200000

float64는 약 \(10^{-308}\) 미만이면 정확히 0.0으로 떨어집니다.
각 단어의 기여가 0.98 수준인 메일에서, 어휘 \(V=2{,}000\)면 아직
살아있지만 \(V=200{,}000\)이면 두 클래스 가능도 모두 0.0이 되어
비교가 불가능해집니다. 로그 공간에는 유한한 숫자가 남습니다.


In [6]:
print(f"0.98^2000    = {0.98**2000:.3e}    <- 아직 살아있다 (약 2.8e-18)")
print(f"0.98^200000  = {0.98**200000}     <- 정확히 0.0, '0.0 vs 0.0' 비교 불가")
print(f"log 공간     = {200000 * math.log(0.98):.1f}   <- 유한한 숫자, 비교 가능")


0.98^2000    = 2.832e-18    <- 아직 살아있다 (약 2.8e-18)
0.98^200000  = 0.0     <- 정확히 0.0, '0.0 vs 0.0' 비교 불가
log 공간     = -4040.5   <- 유한한 숫자, 비교 가능


## 6. 파라미터 수: 나이브베이즈(선형) vs GDA(제곱)

이진 분류의 파라미터 수 — 나이브베이즈 \(2V+1\)(사전확률 1개 +
단어별 등장 확률 \(V\)개 × 클래스 2) vs GDA \(V^2+V\)(공분산 행렬
\(V^2\) + 평균 차 벡터 \(V\)). 어휘가 50,000이면 격차가 약 25,000배.
log-log 그림에서 두 직선의 기울기가 1과 2인 것이 바로 "선형 vs 제곱"
증가의 모습입니다.


In [7]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

# 한글 폰트 (repo 다른 노트북 컨벤션)
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

V = np.array([1_000, 10_000, 50_000])
nb_params  = 2 * V + 1
gda_params = V**2 + V

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))

ax1.bar(V - 0.18 * V, nb_params,  width=0.36 * V,
        label="Naive Bayes $2V+1$", color="#4c72b0")
ax1.bar(V + 0.18 * V, gda_params, width=0.36 * V,
        label="GDA $V^2+V$", color="#dd8452")
ax1.set_yscale("log"); ax1.set_xscale("log")
ax1.set_xticks(V); ax1.set_xticklabels(["1,000", "10,000", "50,000"])
ax1.set_xlabel("어휘 크기 V"); ax1.set_ylabel("파라미터 수 (log)")
ax1.set_title("파라미터 수"); ax1.legend(); ax1.grid(axis="y", alpha=0.3)

Vv = np.logspace(2.5, 5, 200)
ax2.loglog(Vv, 2 * Vv + 1, label="나이브베이즈 $2V+1$ (기울기 1)")
ax2.loglog(Vv, Vv**2 + Vv, label="GDA $V^2+V$ (기울기 2)")
ax2.set_xlabel("어휘 크기 V"); ax2.set_ylabel("파라미터 수")
ax2.set_title("증가 방식: 선형 vs 제곱"); ax2.legend(); ax2.grid(alpha=0.3)

fig.tight_layout()

# 저장 위치: repo 루트에서 올라가서 kor/src/images/ (Colab 등 루트를 못 찾으면 cwd)
p = Path.cwd()
root = None
while p.parent != p:
    if p.name == "book-ml":
        root = p
        break
    p = p.parent
if root is not None:
    out = root / "kor/src/images/ch03_naivebayes_paramcount.svg"
    fig.savefig(out)
    print(f"SVG 저장: {out}")
else:
    fig.savefig("ch03_naivebayes_paramcount.svg")
    print("book-ml 루트를 찾지 못해 cwd에 저장")
plt.show()

for v, a, b in zip(V, nb_params, gda_params):
    print(f"V={v:>7,}  NB={a:>12,}  GDA={b:>15,}  격차 {b/a:,.0f}배")


SVG 저장: /home/smhan/book-ml/kor/src/images/ch03_naivebayes_paramcount.svg
V=  1,000  NB=       2,001  GDA=      1,001,000  격차 500배
V= 10,000  NB=      20,001  GDA=    100,010,000  격차 5,000배
V= 50,000  NB=     100,001  GDA=  2,500,050,000  격차 25,000배


## 7. 정리

- **다항 나이브베이즈**: 학습에서 보지 못한 단어 조합 4통도 4/4 분류,
  margin 3.3~4.1 nats(= 가능도비 약 27~60배)
- **손 계산 검증**: 2-단어 장난감 모델이 본문 표(0.857 / 0.667 /
  0.182 / 0.400)를 그대로 재현
- **스무딩**: 없는 단어 하나만으로도 가능도가 통째로 0이 되지만,
  라플라스 스무딩은 그 단어를 "중립 신호"로 바꿔 나머지 증거(3:1)로
  판단하게 한다
- **사전확률**: 가능도비가 2처럼 동점 근처면 분류를 \(P(y)\)가
  좌우(0.5/0.9/0.1 -> 0.667/0.947/0.182, 스팸 10%에선 뒤집힘)
- **언더플로우**: `0.98**200000 = 0.0` — 그래서 확률의 곱을 만들지
  않고 처음부터 로그만 더한다
- **파라미터 수**: \(2V+1\)(선형) vs \(V^2+V\)(제곱) — 독립 가정이
  차원의 저주를 피하는 수학적 이유

GDA와 나이브베이즈는 모델링 방식(연속값에 정규분포 vs 이산값에 독립
가정)은 다르지만, 둘 다 "먼저 각 클래스가 데이터를 어떻게 만드는지
모델링하고, 베이즈 정리로 뒤집는다"는 같은 생성적 철학을 공유한다.

